# Experiment 4 - Complete Supplementary Measurements

Measured results for the full hyperparameter value set, five-architecture comparison, VGG16 and ResNet50 transfer learning, Adam vs SGD, and frozen vs partial fine-tuning.

In [ ]:
import time, json, random, gc
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
OUT=Path('supplement_outputs'); OUT.mkdir(exist_ok=True)
HP_HIST={}
(x_all,y_all),(x_test_all,y_test_all)=keras.datasets.cifar10.load_data(); y_all=y_all.ravel(); y_test_all=y_test_all.ravel()
x_hp,_,y_hp,_=train_test_split(x_all,y_all,train_size=12000,random_state=SEED,stratify=y_all)
x_hp_train,x_hp_val,y_hp_train,y_hp_val=train_test_split(x_hp,y_hp,test_size=2000,random_state=SEED,stratify=y_hp)
x_hp_test,_,y_hp_test,_=train_test_split(x_test_all,y_test_all,train_size=2000,random_state=SEED,stratify=y_test_all)
x_arch,_,y_arch,_=train_test_split(x_all,y_all,train_size=3000,random_state=SEED,stratify=y_all)
x_arch_train,x_arch_val,y_arch_train,y_arch_val=train_test_split(x_arch,y_arch,test_size=600,random_state=SEED,stratify=y_arch)
x_arch_test,_,y_arch_test,_=train_test_split(x_test_all,y_test_all,train_size=1000,random_state=SEED,stratify=y_test_all)
print('HP:',x_hp_train.shape,x_hp_val.shape,x_hp_test.shape,'ARCH:',x_arch_train.shape,x_arch_val.shape,x_arch_test.shape)

## Hyperparameter Study

In [ ]:
pre=keras.applications.mobilenet_v2.preprocess_input
base=keras.applications.MobileNetV2(weights='imagenet',include_top=False,input_shape=(32,32,3),pooling='avg'); base.trainable=False
ftr=base.predict(pre(x_hp_train.astype('float32')),batch_size=256,verbose=0)
fva=base.predict(pre(x_hp_val.astype('float32')),batch_size=256,verbose=0)
fte=base.predict(pre(x_hp_test.astype('float32')),batch_size=256,verbose=0)
def make_head(units=128,opt='Adam',lr=0.001):
    m=keras.Sequential([layers.Input((ftr.shape[1],)),layers.Dense(units,activation='relu'),layers.Dropout(0.2),layers.Dense(10,activation='softmax')])
    o=keras.optimizers.Adam(lr) if opt=='Adam' else keras.optimizers.SGD(lr,momentum=0.9)
    m.compile(optimizer=o,loss='sparse_categorical_crossentropy',metrics=['accuracy']); return m
def hp_run(label,lr,batch,epochs,opt,units):
    m=make_head(units,opt,lr); t=time.perf_counter(); h=m.fit(ftr,y_hp_train,validation_data=(fva,y_hp_val),epochs=epochs,batch_size=batch,verbose=0); sec=time.perf_counter()-t
    _,acc=m.evaluate(fte,y_hp_test,batch_size=256,verbose=0)
    HP_HIST[label]={'accuracy':[float(v) for v in h.history['accuracy']], 'val_accuracy':[float(v) for v in h.history['val_accuracy']], 'loss':[float(v) for v in h.history['loss']], 'val_loss':[float(v) for v in h.history['val_loss']]}
    r={'Setting':label,'Learning Rate':lr,'Batch Size':batch,'Epochs':epochs,'Optimizer':opt,'Dense Units':units,'Frozen Layers':'All','Best Validation Accuracy':float(max(h.history['val_accuracy'])),'Test Accuracy':float(acc),'Training Time (s)':float(sec)}
    del m; gc.collect(); return r
rows=[hp_run('Baseline',.001,32,10,'Adam',128),hp_run('Learning rate 0.0001',.0001,32,10,'Adam',128),hp_run('Batch size 16',.001,16,10,'Adam',128),hp_run('Batch size 64',.001,64,10,'Adam',128),hp_run('Epochs 20',.001,32,20,'Adam',128),hp_run('Optimizer SGD',.001,32,10,'SGD',128),hp_run('Dense units 256',.001,32,10,'Adam',256)]
hp_df=pd.DataFrame(rows); hp_df.to_csv(OUT/'hyperparameter_study_complete.csv',index=False); hp_df

## Five-Architecture Comparison

In [ ]:
xn=x_arch_train.astype('float32')/255.; xv=x_arch_val.astype('float32')/255.; xt=x_arch_test.astype('float32')/255.
def build_lenet():
    i=keras.Input((32,32,3)); x=layers.Conv2D(6,5,activation='tanh')(i); x=layers.AveragePooling2D(2)(x); x=layers.Conv2D(16,5,activation='tanh')(x); x=layers.AveragePooling2D(2)(x); x=layers.Flatten()(x); x=layers.Dense(120,activation='tanh')(x); x=layers.Dense(84,activation='tanh')(x); return keras.Model(i,layers.Dense(10,activation='softmax')(x))
def build_alexnet():
    i=keras.Input((32,32,3)); x=layers.Conv2D(64,3,padding='same',activation='relu')(i); x=layers.MaxPooling2D(2)(x); x=layers.Conv2D(192,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D(2)(x); x=layers.Conv2D(384,3,padding='same',activation='relu')(x); x=layers.Conv2D(256,3,padding='same',activation='relu')(x); x=layers.Conv2D(256,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D(2)(x); x=layers.GlobalAveragePooling2D()(x); x=layers.Dense(512,activation='relu')(x); return keras.Model(i,layers.Dense(10,activation='softmax')(x))
def inc(x,f):
    a=layers.Conv2D(f,1,padding='same',activation='relu')(x); b=layers.Conv2D(f,1,padding='same',activation='relu')(x); b=layers.Conv2D(f,3,padding='same',activation='relu')(b); d=layers.MaxPooling2D(3,strides=1,padding='same')(x); d=layers.Conv2D(f//2,1,padding='same',activation='relu')(d); return layers.Concatenate()([a,b,d])
def build_googlenet():
    i=keras.Input((32,32,3)); x=layers.Conv2D(64,3,padding='same',activation='relu')(i); x=layers.MaxPooling2D(2)(x); x=inc(x,64); x=inc(x,96); x=layers.MaxPooling2D(2)(x); x=inc(x,128); x=layers.GlobalAveragePooling2D()(x); return keras.Model(i,layers.Dense(10,activation='softmax')(x))
def scratch(name,builder):
    tf.keras.backend.clear_session(); m=builder(); m.compile(optimizer=keras.optimizers.Adam(.001),loss='sparse_categorical_crossentropy',metrics=['accuracy']); t=time.perf_counter(); m.fit(xn,y_arch_train,validation_data=(xv,y_arch_val),epochs=2,batch_size=64,verbose=2); sec=time.perf_counter()-t; _,acc=m.evaluate(xt,y_arch_test,batch_size=256,verbose=0); r={'Model':name,'Parameters':int(m.count_params()),'Accuracy':float(acc),'Training Time (s)':float(sec)}; del m; gc.collect(); return r
arch=[scratch('LeNet-5',build_lenet),scratch('AlexNet',build_alexnet),scratch('GoogleNet',build_googlenet)]

## VGG16 and ResNet50 Transfer Learning

In [ ]:
def pretrained_compare(builder,preprocess,name,unfreeze):
    tf.keras.backend.clear_session(); b=builder(weights='imagenet',include_top=False,input_shape=(32,32,3),pooling='avg'); b.trainable=False
    i=keras.Input((32,32,3)); x=preprocess(i); x=b(x,training=False); x=layers.Dense(128,activation='relu')(x); o=layers.Dense(10,activation='softmax')(x); m=keras.Model(i,o); m.compile(optimizer=keras.optimizers.Adam(.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    t=time.perf_counter(); hf=m.fit(x_arch_train,y_arch_train,validation_data=(x_arch_val,y_arch_val),epochs=1,batch_size=64,verbose=2); _,fa=m.evaluate(x_arch_test,y_arch_test,batch_size=256,verbose=0)
    b.trainable=True
    for l in b.layers[:-unfreeze]: l.trainable=False
    for l in b.layers[-unfreeze:]:
        if isinstance(l,layers.BatchNormalization): l.trainable=False
    m.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy']); ht=m.fit(x_arch_train,y_arch_train,validation_data=(x_arch_val,y_arch_val),epochs=1,batch_size=64,verbose=2); sec=time.perf_counter()-t; pred=m.predict(x_arch_test,batch_size=256,verbose=0).argmax(1); acc=accuracy_score(y_arch_test,pred); p,r,f,_=precision_recall_fscore_support(y_arch_test,pred,average='weighted',zero_division=0); cm=confusion_matrix(y_arch_test,pred)
    out={'Model':name,'Parameters':int(m.count_params()),'Accuracy':float(acc),'Training Time (s)':float(sec),'Frozen Accuracy':float(fa),'Fine-Tuned Accuracy':float(acc),'Precision':float(p),'Recall':float(r),'F1-score':float(f),'Frozen Training Accuracy':float(hf.history['accuracy'][-1]),'Frozen Validation Accuracy':float(hf.history['val_accuracy'][-1]),'Fine-Tune Training Accuracy':float(ht.history['accuracy'][-1]),'Fine-Tune Validation Accuracy':float(ht.history['val_accuracy'][-1]),'Confusion Matrix':cm.tolist(),'Frozen History':{'accuracy':[float(v) for v in hf.history['accuracy']],'val_accuracy':[float(v) for v in hf.history['val_accuracy']],'loss':[float(v) for v in hf.history['loss']],'val_loss':[float(v) for v in hf.history['val_loss']]},'Fine-Tune History':{'accuracy':[float(v) for v in ht.history['accuracy']],'val_accuracy':[float(v) for v in ht.history['val_accuracy']],'loss':[float(v) for v in ht.history['loss']],'val_loss':[float(v) for v in ht.history['val_loss']}}; del m,b; gc.collect(); return out
vgg=pretrained_compare(keras.applications.VGG16,keras.applications.vgg16.preprocess_input,'VGG16',4); res=pretrained_compare(keras.applications.ResNet50,keras.applications.resnet50.preprocess_input,'ResNet50',10)
arch += [{k:v for k,v in vgg.items() if k in ['Model','Parameters','Accuracy','Training Time (s)']},{k:v for k,v in res.items() if k in ['Model','Parameters','Accuracy','Training Time (s)']}]
arch_df=pd.DataFrame(arch); arch_df['Accuracy (%)']=arch_df['Accuracy']*100; arch_df.to_csv(OUT/'architecture_comparison.csv',index=False); arch_df

## Frozen vs Partial Fine-Tuning

In [ ]:
tf.keras.backend.clear_session(); mb=keras.applications.MobileNetV2(weights='imagenet',include_top=False,input_shape=(32,32,3),pooling='avg'); mb.trainable=False
i=keras.Input((32,32,3)); x=keras.applications.mobilenet_v2.preprocess_input(i); x=mb(x,training=False); x=layers.Dense(128,activation='relu')(x); o=layers.Dense(10,activation='softmax')(x); mm=keras.Model(i,o); mm.compile(optimizer=keras.optimizers.Adam(.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
t=time.perf_counter(); hfr=mm.fit(x_hp_train,y_hp_train,validation_data=(x_hp_val,y_hp_val),epochs=3,batch_size=32,verbose=2); ft=time.perf_counter()-t; _,facc=mm.evaluate(x_hp_test,y_hp_test,batch_size=256,verbose=0)
mb.trainable=True
for l in mb.layers[:-20]: l.trainable=False
for l in mb.layers[-20:]:
    if isinstance(l,layers.BatchNormalization): l.trainable=False
mm.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy']); t=time.perf_counter(); hpt=mm.fit(x_hp_train,y_hp_train,validation_data=(x_hp_val,y_hp_val),epochs=2,batch_size=32,verbose=2); pt=time.perf_counter()-t; _,pacc=mm.evaluate(x_hp_test,y_hp_test,batch_size=256,verbose=0)
frozen_partial=pd.DataFrame([{'Training Strategy':'Frozen convolutional base','Test Accuracy':float(facc),'Training Time (s)':float(ft)},{'Training Strategy':'Partial fine-tuning','Test Accuracy':float(pacc),'Training Time (s)':float(pt)}]); frozen_partial.to_csv(OUT/'frozen_vs_partial.csv',index=False)
hp_df2=pd.concat([hp_df,pd.DataFrame([{'Setting':'Frozen layers partial','Learning Rate':1e-5,'Batch Size':32,'Epochs':2,'Optimizer':'Adam','Dense Units':128,'Frozen Layers':'Partial','Best Validation Accuracy':float(max(hpt.history['val_accuracy'])),'Test Accuracy':float(pacc),'Training Time (s)':float(pt)}])],ignore_index=True); hp_df2.to_csv(OUT/'hyperparameter_study_complete.csv',index=False); frozen_partial

In [ ]:
all_results={'hyperparameter_histories':HP_HIST,'hyperparameter_study':hp_df2.to_dict(orient='records'),'architecture_comparison':arch_df.to_dict(orient='records'),'vgg16':vgg,'resnet50':res,'adam_vs_sgd':hp_df2[hp_df2['Setting'].isin(['Baseline','Optimizer SGD'])].to_dict(orient='records'),'frozen_vs_partial':frozen_partial.to_dict(orient='records')}
with open(OUT/'supplement_results.json','w') as f: json.dump(all_results,f,indent=2)
print(json.dumps(all_results,indent=2))